In [1]:
from pyspark import SparkContext
import re
from datetime import datetime

In [2]:
sc = SparkContext("local", "LogParser")
lines = sc.textFile("hdfs:///data/dadakhon/access.log")
log_pattern = re.compile(
    r'(?P<remote_addr>\S+) - - \[(?P<time_local>.*?)\] "(?P<request>\S+) (?P<url>\S+)[^"]*" (?P<status>\d{3}) (?P<body_bytes_sent>\d+|-)' 
)

def parse_line(line):
    match = log_pattern.match(line)
    if match:
        time_str = match.group("time_local").split()[0]
        dt = datetime.strptime(time_str, "%d/%b/%Y:%H:%M:%S")
        return {
            "remote_addr": match.group("remote_addr"),
            "time_local": {
                "year": dt.year,
                "month": dt.month,
                "day": dt.day,
                "hours": dt.hour,
                "minutes": dt.minute,
                "seconds": dt.second
            },
            "request": match.group("request"),
            "url": match.group("url"),
            "status": int(match.group("status")),
            "body_bytes_sent": int(match.group("body_bytes_sent")) if match.group("body_bytes_sent").isdigit() else 0
        }
parsed_rdd = lines.map(parse_line).filter(lambda x: x is not None)